
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lecture - Types of Evaluation Judges

## Overview

MLflow provides multiple types of evaluation judges, each designed for different evaluation scenarios and customization levels. This lecture explores the spectrum of available judges, from built-in research-validated assessments to fully custom evaluation logic.

We'll examine built-in judges for common criteria, guideline judges for business rules, and custom approaches for specialized requirements. Understanding when and how to use each type is crucial for building comprehensive evaluation workflows.

**Learning Objectives**

By the end of this lecture, you will be able to:
- Distinguish between different types of evaluation judges (built-in, guideline, custom)
- Identify appropriate built-in judges for common evaluation criteria
- Understand how to implement guideline judges for business rules
- Recognize when custom judges are necessary and how to implement them
- Understand the role of Feedback objects and rationales in evaluation results

## A. Judge Types Overview

### A1. Evaluation Judge Spectrum

| Approach              | Level of customization | Use cases |
|----------------------|------------------------|-----------|
| Built-in judges      | Minimal                | Quickly try LLM evaluation with built-in scorers such as `Correctness` and `RetrievalGroundedness`. |
| Guidelines judges    | Moderate               | A built-in judge that checks whether responses pass or fail custom natural-language rules, such as style or factuality guidelines. |
| Custom judges        | Full                   | Create fully customized LLM judges with detailed evaluation criteria and feedback optimization. Capable of returning numerical scores, categories, or boolean values. |
| Code-based scorers   | Full                   | Programmatic and deterministic scorers that evaluate things like exact matching, format validation, and performance metrics. |

## B. Built-In Judges for Common Criteria

### B1. Research-Validated Judges

MLflow provides research-validated judges for common evaluation tasks. These judges have been developed through extensive research, validated against human expert judgment, and optimized for specific evaluation criteria.

### B2. Core Built-in Judges

**Correctness**  
Evaluates whether the model's response is factually correct compared to provided ground truth.  
Requires ground truth in the evaluation dataset via `expectations` (for example, an expected answer or expected facts).

**RelevanceToQuery**  
Assesses whether the response directly and appropriately addresses the user's query.  
Useful for identifying off-topic, tangential, or irrelevant answers. Does **not** require ground truth.

**RetrievalSufficiency**  
Determines whether the retrieved context contains all the information necessary to produce a correct response that includes the ground-truth facts.  
Requires ground truth (`expectations`) and evaluates retrieval quality rather than generation quality.

**RetrievalRelevance**  
Evaluates whether the retrieved documents are relevant to the user's query.  
Does **not** require ground truth and focuses only on the retrieval step, independent of the final answer.

### B3. Additional Built-in Judges

**RetrievalGroundedness**  
Checks whether the model's response is grounded in the retrieved context and does not hallucinate unsupported facts.  
Does **not** require ground truth and evaluates alignment between the response and provided documents.

**Safety**  
Assesses whether the response is free from harmful, offensive, or unsafe content.  
Does **not** require ground truth and is commonly used as a baseline content safety check.

**Guidelines**  
Evaluates whether the response follows specified natural-language rules or constraints (for example, style, tone, or formatting requirements).  
Does **not** require ground truth.

**ExpectationsGuidelines**  
Evaluates whether the response meets per-example natural-language criteria defined in the evaluation dataset.  
Does not require factual ground truth, but relies on example-specific guidelines provided in `expectations`.

### B4. Example Usage Pattern

```python
from mlflow.genai.scorers import Correctness

correctness_eval = Correctness(
    model="databricks:/foundation-model-endpoint")

correctness_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: agent.predict({"input": input}),
    scorers=[correctness_eval],
)
```

This example demonstrates a common pattern for evaluating agents using the `Correctness` scorer with a Databricks foundation model endpoint.

## ⚠️ Demo Checkpoint
<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;"></span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">Demo Checkpoint</strong>
      <p style="margin: 8px 0 0 0; color: #333;">Navigate to <strong>2.2 Demo - Using MLflow Built-In Judges</strong> to see these built-in judges in action. When you are finished, navigate back to this lecture notebook and continue learning.</p>
    </div>
  </div>
</div>

## C. Guideline Judges

### C1. Two Types of Guideline Judges

**1. Global Guidelines (`Guidelines` class)**

Apply uniform criteria to all evaluations in your dataset. Global guidelines are ideal when you want to enforce consistent standards across all test cases, such as tone, style, or formatting requirements.

```python
from mlflow.genai.scorers import Guidelines

tone_guidelines = Guidelines(
    name="professional_tone",
    guidelines=[
        "The response must use professional, business-appropriate language",
        "The response should avoid slang, colloquialisms, or overly casual phrasing",
        "The response must address the user respectfully"
    ],
    model="databricks:/foundation-model-endpoint"
)
```

### C2. Per-Row Guidelines

**2. Per-Row Guidelines (`ExpectationsGuidelines` class)**

Apply different criteria to each example, useful when different scenarios require different standards. Each row in your dataset includes its own specific guidelines in an `expectations` field.

```python
from mlflow.genai.scorers import ExpectationsGuidelines

# Dataset contains per-row guidelines in expectations
# Example row:
# {
#   "input": "What's the refund policy?",
#   "output": "Our refund policy allows...",
#   "expectations": {
#     "guidelines": ["Must mention 30-day timeframe", "Must include receipt requirement"]
#   }
# }

expected_guidelines = ExpectationsGuidelines(
    name="policy_requirements",
    model="databricks:/foundation-model-endpoint"
)
```

This approach is ideal for testing diverse use cases where each input requires unique validation criteria.

### C3. Guideline Judge Advantages and Best Practices

**Guideline judge advantages:**

- **Domain expert accessibility**: Business stakeholders can write guidelines without coding
- **Rapid iteration**: Update evaluation criteria without code changes
- **Interpretability**: Natural language guidelines are self-documenting
- **Flexibility**: Express complex, context-dependent requirements that would be difficult to code

**Tips for writing guidelines:**

- Be specific and concrete rather than vague ("Response must cite the source document" vs. "Response should be credible")
- Write guidelines that can be objectively verified
- Focus on observable attributes in the response
- Test guidelines on multiple examples to ensure they work as intended

## ⚠️ Demo Checkpoint
<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;"></span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">Demo Checkpoint</strong>
      <p style="margin: 8px 0 0 0; color: #333;">Navigate to <strong>2.3 Demo - Guideline Judges with MLflow</strong> to explore guideline judges in practice. When you are finished, navigate back to this lecture notebook and continue learning.</p>
    </div>
  </div>
</div>

## ⚠️ Lab Checkpoint
<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;"></span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">Lab Checkpoint</strong>
      <p style="margin: 8px 0 0 0; color: #333;">Navigate to <strong>2.4 Lab - Applying Agent Evaluation</strong> to practice implementing comprehensive evaluation workflows. When you are finished, navigate back to this lecture notebook and continue learning.</p>
    </div>
  </div>
</div>

## D. Custom Judges and Code-Based Scorers

### D1. Code-Based Scorers for Deterministic Evaluation

When built-in judges don't meet your needs, MLflow supports custom evaluation logic through code-based scorers:

```python
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback

@scorer
def response_length(outputs):
    """Verify response length is appropriate."""
    word_count = len(str(outputs.get("response", "")).split())

    if 20 <= word_count <= 100:
        return Feedback(
            value="yes",
            rationale=f"Response length ({word_count} words) is appropriate"
        )
    else:
        return Feedback(
            value="no",
            rationale=f"Response is too {'short' if word_count < 20 else 'long'} ({word_count} words)"
        )
```

Custom scorers allow you to implement domain-specific validation logic that goes beyond what built-in judges provide. The `@scorer` decorator converts your function into a reusable evaluation component.

### D2. Alternative Approach with Primitive Return

For simpler use cases, scorers can return primitive values directly:

```python
from mlflow.genai.scorers import scorer

@scorer
def response_length(outputs):
    wc = len(str(outputs.get("response", "")).split())
    return "yes" if 20 <= wc <= 100 else "no"
```

This approach is more concise but doesn't provide rationale for the scoring decision, making it better suited for straightforward pass/fail checks.

`@scorer` turns your plain Python function into an MLflow GenAI Scorer, which is a first-class, pluggable metric that `mlflow.genai.evaluate()` can run offline and that you can register for production monitoring later.

### D3. Custom LLM Judges

**Custom LLM judges** for sophisticated evaluation:

You can implement your own LLM-based judges for specialized evaluation criteria not covered by built-in judges. This involves:
1. Designing evaluation prompts that clearly specify your criteria
2. Calling an LLM to make judgments based on those prompts
3. Parsing LLM responses into structured `Feedback` objects
4. Wrapping this logic in a function compatible with `mlflow.genai.evaluate()`

**When to use custom judges:**

- You need domain-specific evaluation criteria unique to your application
- Built-in judges don't capture nuances important to your use case
- You're evaluating against proprietary standards or regulations
- You need evaluation logic that combines multiple signals or data sources

Custom judges provide maximum flexibility while maintaining integration with MLflow's evaluation framework, tracing, and logging infrastructure.

## ⚠️ Demo Checkpoint
<div style="border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <span style="font-size: 24px;"></span>
    <div>
      <strong style="color: #e65100; font-size: 1.1em;">Demo Checkpoint</strong>
      <p style="margin: 8px 0 0 0; color: #333;">Navigate to <strong>2.5 Demo - Custom Judges with MLflow</strong> to learn how to create custom judges for specialized evaluation needs. When you are finished, navigate back to this lecture notebook and continue learning.</p>
    </div>
  </div>
</div>

## E. Feedback Objects and Rationales

### E1. Structured Feedback Objects

```python
Feedback(
    value="yes",        # Binary pass/fail or numerical score
    rationale="The response correctly identifies the capital as Sacramento...",  # Explanation
    metadata={...}      # Additional structured information
)
```

MLflow judges return structured `Feedback` objects rather than simple scalar scores. This structure provides interpretability critical for understanding evaluation results.

**Key components:**
- `value`: The actual score or judgment
- `rationale`: Human-readable explanation
- `metadata`: Additional context

### E2. Why Rationales Matter

- **Debugging**: Understand why an example failed, not just that it failed
- **Judge validation**: Verify that judges are reasoning correctly
- **Pattern identification**: Common rationale themes reveal systemic issues
- **Stakeholder communication**: Explain evaluation results to non-technical audiences

**Using rationales effectively:**

1. Read rationales for all failures to identify patterns
2. Spot-check rationales for passes to ensure judge is reasoning correctly
3. Extract common rationale phrases to categorize failure types
4. Share representative rationales when discussing evaluation with teams

The combination of pass/fail scores with detailed rationales makes MLflow evaluation both quantitative (trackable metrics) and qualitative (understandable reasoning).

## F. Key Takeaways

MLflow provides a comprehensive suite of evaluation judges to meet diverse evaluation needs:

1. **Built-in judges**: Research-validated assessments for common criteria like correctness, relevance, and safety
2. **Guideline judges**: Business-rule evaluation through natural language guidelines, both global and per-example
3. **Custom code-based scorers**: Deterministic evaluation logic for specific requirements
4. **Custom LLM judges**: Sophisticated evaluation for specialized domain requirements
5. **Structured feedback**: Rationales provide interpretability and debugging capabilities

Choosing the right combination of judges depends on your specific evaluation requirements, available ground truth, and the level of customization needed. The next lectures will explore evaluation strategies and practical implementation approaches.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>